# Django Class-based Views

Alumno: Héctor Hugo Hernández

## Objetivo

Crear un ListView basado en clases protegido mediante LoginRequiredMixin que muestre únicamente los productos correspondientes al usuario autenticado.

## ProtectedListView

Código real de `src/ecommerce/views.py`:

```python
class ProtectedListView(LoginRequiredMixin, ListView):
    model = ProductModel
    template_name = "ecommerce/list-view.html"
    context_object_name = "products"
    login_url = "ecommerce-login"

    def get_queryset(self):
        return ProductModel.objects.filter(owner=self.request.user)
```

- `ListView` obtiene los objetos y renderiza el listado.
- `LoginRequiredMixin` exige una sesión autenticada antes de ejecutar la vista.
- `get_queryset()` define los productos disponibles en esta vista.
- `self.request.user` representa al usuario que realiza la solicitud.

Se reutiliza `ecommerce/list-view.html`, que ya recorre la variable `products`.

## URL my-products

Registro real en `src/ecommerce/urls.py`:

```python
path("my-products/", views.ProtectedListView.as_view(), name="my-products"),
```

El proyecto incluye ecommerce bajo `/ecommerce/`. URL de prueba: `http://localhost:8000/ecommerce/my-products/`.

## Relación entre producto y usuario

ProductModel no contenía una relación con el usuario. Se agregó este campo y el import `from django.conf import settings`:

```python
owner = models.ForeignKey(
        settings.AUTH_USER_MODEL,
        on_delete=models.CASCADE,
        related_name="products",
        null=True,
        blank=True,
    )
```

`owner` identifica al propietario mediante `settings.AUTH_USER_MODEL`. `null=True` y `blank=True` mantienen compatibles los productos anteriores y el fixture de 500 productos. Los productos sin propietario siguen en el listado general, pero no aparecen en my-products. La migración agrega el campo sin asignar propietarios ni borrar registros.

### Creación e inicio de sesión

La vista `product_model_create_view` usa `@login_required(login_url="ecommerce-login")` y asigna `owner=request.user` al construir el producto. Así evita asignar un usuario anónimo o aceptar un propietario enviado por el formulario.

`/ecommerce/login/` reutiliza `django.contrib.auth.views.LoginView`, con el template `ecommerce/login.html` y `next_page="my-products"`. Permite iniciar sesión con usuarios existentes, incluidos los que no son administradores, y conserva el destino `next`. No se crea un sistema nuevo de autenticación.

## Resultado

- Un visitante no autenticado recibe una redirección al login con el parámetro `next`; no accede al listado protegido.
- Un usuario autenticado puede acceder a my-products.
- `get_queryset()` limita los resultados a productos cuyo `owner` corresponde a `request.user`.
- Un usuario sin productos obtiene un listado vacío.
- El listado general y las vistas existentes se conservan; la creación ahora requiere autenticación. El filtrado de esta actividad se aplica al listado my-products.

### Aplicar la migración en Docker

Desde la raíz del repositorio, con Docker y los servicios del proyecto disponibles:

```console
docker compose build web
docker compose run --rm --no-deps --entrypoint python web manage.py migrate
docker compose up -d --no-deps web
docker compose exec web python manage.py check
```

Se reconstruye la imagen porque el Dockerfile copia el código. Estos comandos se documentan para ejecución manual; el notebook no los ejecuta. No es necesario volver a cargar el fixture.

### Validación realizada

Se generó `0003_productmodel_owner.py` y no hay cambios de modelos pendientes. Pasaron 8 pruebas de ecommerce con SQLite en memoria. Otra comprobación cargó los 500 productos del fixture en el esquema anterior y aplicó la migración: todos los registros y campos originales permanecieron iguales, con `owner=NULL`. El hash SHA-256 del fixture original no cambió.

Django check no encontró errores; en Windows indicó la advertencia preexistente staticfiles.W004 por la ruta Docker `/public`. Las pruebas usaron rutas locales de archivos estáticos y caché en memoria mediante ajustes temporales del proceso. No se modificaron las configuraciones del proyecto ni se accedió a PostgreSQL. Docker no estaba disponible; la migración de la base real queda pendiente de ejecución manual.

## Conclusión

Las Class-Based Views permiten reutilizar comportamiento de Django. ListView simplifica la presentación de colecciones, LoginRequiredMixin exige autenticación y get_queryset permite filtrar los datos según el usuario de la solicitud. Una relación real con el usuario hace posible identificar sus productos.

Este notebook solo documenta el trabajo: no ejecuta código ni modifica bases de datos.